In [ ]:
from google.colab import drive
import os

# 1. 드라이브 마운트
drive.mount('/content/drive')

# 2. 작업 경로 설정 및 이동
base_path = "/content/drive/MyDrive/Mecro" # 태영 님의 폴더 경로
if not os.path.exists(base_path):
    os.makedirs(base_path)

%cd {base_path}
print(f"현재 작업 경로: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Mecro
현재 작업 경로: /content/drive/MyDrive/Mecro


In [ ]:
!git checkout -b feature/data-factory

Switched to a new branch 'feature/data-factory'


In [ ]:
# 필요한 라이브러리 설치
!pip install yfinance beautifulsoup4 pandas requests

import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import os

# 데이터 저장 경로 설정
DATA_PATH = "./data/raw"
if not os.path.exists(DATA_PATH):
  os.makedirs(DATA_PATH)

In [ ]:
# 거시경제 지표 및 원자재 시세 수집 (yfinance)

def fetch_market_data():
    # 수집 대상 티커 (국채금리, 금, 구리, 달러인덱스, 알루미늄)
    tickers = {
        "10Y_Bond": "^TNX",
        "Gold": "GC=F",
        "Copper": "HG=F",
        "USD_Index": "DX-Y.NYB",
        "Aluminum": "ALI=F"
    }

    market_df = pd.DataFrame()

    for name, ticker in tickers.items():
        print(f"{name} 데이터 수집 중...")
        asset = yf.Ticker(ticker)
        # RAG와의 시점 동기화를 위해 1시간 단위(1h)로 수집
        df = asset.history(period="1mo", interval="1h")

        if df.empty:
            print(f"경고: {name} 데이터를 가져올 수 없습니다.")
            continue

        # 종가(Close) 기준으로 병합
        market_df[name] = df['Close']

    market_df.to_csv(f"{DATA_PATH}/market_prices.csv")
    print("시세 데이터 저장 완료.")
    return market_df

# 실행
market_prices = fetch_market_data()

10Y_Bond 데이터 수집 중...
Gold 데이터 수집 중...
Copper 데이터 수집 중...
USD_Index 데이터 수집 중...
Aluminum 데이터 수집 중...
시세 데이터 저장 완료.


In [ ]:
# 야후 파이낸셜 뉴스 크롤링 (BeautifulSoup)

def fetch_yahoo_news_v2(topic="economic-news"):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # 쿼리 기반의 검색 페이지보다 '토픽' 페이지가 더 안정적인 구조를 가집니다.
    # 예: economic-news, stock-market-news 등
    url = f"https://finance.yahoo.com/topic/{topic}/"

    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')

        news_list = []

        # 최근 야후 파이낸셜은 h3 태그와 그 부모 링크(a) 구조를 유지하고 있습니다.
        # 모든 h3 태그 중 뉴스 제목일 가능성이 높은 것들을 추출합니다.
        articles = soup.find_all('h3')

        for title_tag in articles:
            # h3 태그의 부모나 자식 중에 <a> 태그가 있는지 확인
            link_tag = title_tag.find_parent('a') or title_tag.find('a')

            if link_tag and link_tag.get('href'):
                title = title_tag.get_text(strip=True)
                link = link_tag['href']

                # 상대 경로인 경우 절대 경로로 변환
                if link.startswith('/'):
                    link = "https://finance.yahoo.com" + link

                # 중복 제거 및 광고성 링크 필터링 (간단한 로직)
                if "video" not in link and len(title) > 10:
                    news_list.append({
                        "title": title,
                        "url": link,
                        "collected_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        "source": "Yahoo Finance"
                    })

        # 데이터프레임 변환 및 중복 제거
        news_df = pd.DataFrame(news_list).drop_duplicates(subset=['title'])

        if not news_df.empty:
            news_df.to_csv(f"{DATA_PATH}/raw_news.csv", index=False, encoding='utf-8-sig')
            print(f"성공: {len(news_df)}개의 뉴스 데이터를 수집했습니다.")
        else:
            print("경고: 수집된 뉴스가 없습니다. 셀렉터나 URL을 다시 확인해야 합니다.")

        return news_df

    except Exception as e:
        print(f"오류 발생: {e}")
        return pd.DataFrame()

# 실행 (주제별로 테스트 가능: 'economic-news' 또는 'stock-market-news')
news_data = fetch_yahoo_news_v2("economic-news")

성공: 58개의 뉴스 데이터를 수집했습니다.


In [ ]:
!git add 01_Data_Factory.ipynb
!git commit -m "fix: 야후 파이낸셜 뉴스 크롤러 셀렉터 보정"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@afc61cc9f2f8.(none)')


In [ ]:
from google.colab import userdata

# 1. 깃허브 개인 정보 설정 (본인의 정보로 수정하기)
GIT_USER_NAME = "kimTaeYoungM"
GIT_USER_EMAIL = "taeyeonggim069@gmail.com"

# 2. Secrets에서 토큰 불러오기 (이름이 GH_TOKEN 이어야 한다)
try:
  GIT_TOKEN = userdata.get('GH_TOKEN')
  print("깃허브 토큰을 성공적으로 불러옴.")
except Exception as e:
  print("Secrets에서 GH_TOKEN을 찾을 수 없음. 설정 재확인.")

# 3. 전역 설정 반영
!git config --global user.name "{GIT_USER_NAME}"
!git config --global user.email "{GIT_USER_EMAIL}"

깃허브 토큰을 성공적으로 불러옴.


In [ ]:
GIT_ORG = "DMU-Mecro"
GIT_REPO = "Mecro"

# 현재 폴더에 .git 설정이 있는지 확인
if not os.path.exists(os.path.join(BASE_PATH, ".git")):
  # 처음 가져오는 경우 (Clone)
  # 현재 폴더(.)에 바로 복제하기 위해 뒤에 점(.)을 붙인다.
  !git clone https://{GIT_TOKEN}@github.com/{GIT_ORG}/{GIT_REPO}.git .
  print("저장소를 새로 복제함.")
else:
  # 이미 존재하는 경우 (최신화)
  !git pull origin main
  print("최신 코드로 동기화되었음.")

In [ ]:
# 현재 체크아웃된 브랜치 이름을 가져옵니다.
current_branch = !git rev-parse --abbrev-ref HEAD
current_branch = current_branch[0]

BASE_PATH = "/content/drive/MyDrive/Mecro"

if not os.path.exists(os.path.join(BASE_PATH, ".git")):
    # 처음 가져오는 경우
    !git clone https://{GIT_TOKEN}@github.com/{GIT_ORG}/{GIT_REPO}.git .
    print(f"🚀 저장소를 새로 복제했습니다.")
else:
    # 이미 존재하는 경우: 현재 브랜치에 맞춰 최신화
    !git pull origin {current_branch}
    print(f"✅ 현재 브랜치({current_branch})의 최신 코드로 동기화되었습니다.")

fatal: couldn't find remote ref feature/data-factory
✅ 현재 브랜치(feature/data-factory)의 최신 코드로 동기화되었습니다.


In [ ]:
!git push -u origin feature/data-factory

Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 4.50 KiB | 383.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
remote: 
remote: Create a pull request for 'feature/data-factory' on GitHub by visiting:
remote:      https://github.com/DMU-Mecro/Mecro/pull/new/feature/data-factory
remote: 
To https://github.com/DMU-Mecro/Mecro.git
 * [new branch]      feature/data-factory -> feature/data-factory
Branch 'feature/data-factory' set up to track remote branch 'feature/data-factory' from 'origin'.


In [ ]:
# 현재 브랜치 확인
current_branch = !git rev-parse --abbrev-ref HEAD
current_branch = current_branch[0]

BASE_PATH = "/content/drive/MyDrive/Mecro"

if not os.path.exists(os.path.join(BASE_PATH, ".git")):
    !git clone https://{GIT_TOKEN}@github.com/{GIT_ORG}/{GIT_REPO}.git .
    print("🚀 저장소를 새로 복제했습니다.")
else:
    # pull 시도 후 결과를 변수에 담음
    result = !git pull origin {current_branch}

    # 결과 메시지에 fatal이 포함되어 있는지 확인
    if any("fatal" in line for line in result):
        print(f"⚠️  원격에 '{current_branch}' 브랜치가 아직 없습니다. 첫 push가 필요합니다.")
    else:
        print(f"✅ 현재 브랜치({current_branch})의 최신 코드로 동기화되었습니다.")

✅ 현재 브랜치(feature/data-factory)의 최신 코드로 동기화되었습니다.


In [ ]:
def fetch_robust_market_data():
    tickers = {
        "10Y_Bond": "^TNX",
        "Gold": "GC=F",
        "Copper": "HG=F",
        "USD_Index": "DX-Y.NYB",
        "Aluminum": "ALI=F" # 데이터가 부족할 경우 'LAL=F'로 시도해보세요.
    }

    all_data = []

    for name, ticker in tickers.items():
        print(f"{name} ({ticker}) 수집 중...")
        asset = yf.Ticker(ticker)
        # 1시간 단위 데이터를 가져올 때, 시간대(Timezone) 문제를 방지하기 위해 로컬 시간 제거
        df = asset.history(period="1mo", interval="1h")

        if not df.empty:
            df = df[['Close']].rename(columns={'Close': name})
            # 인덱스를 단순 날짜 형식으로 변환하여 병합 시 충돌 방지
            df.index = df.index.tz_localize(None)
            all_data.append(df)
        else:
            print(f"⚠️ {name} 데이터가 비어있습니다.")

    # 1. 모든 데이터를 시간 축 기준으로 통합 (Outer Join)
    market_df = pd.concat(all_data, axis=1)

    # 2. 결측치 처리 (중요!)
    # 앞선 데이터를 뒤로 밀어서 채움 (시장이 닫혔을 때 직전 가격 유지)
    market_df = market_df.ffill().bfill()

    # 3. 인덱스 이름 정리 및 저장
    market_df.index.name = "Datetime"
    market_df.to_csv(f"{DATA_PATH}/market_prices.csv")

    print(f"\n✅ 수집 완료: {len(market_df)} 행의 데이터가 저장되었습니다.")
    return market_df

# 실행
market_prices = fetch_robust_market_data()
print(market_prices.head(10)) # 이제 Gold나 Copper 칸이 채워졌는지 확인해보세요.

10Y_Bond (^TNX) 수집 중...
Gold (GC=F) 수집 중...
Copper (HG=F) 수집 중...
USD_Index (DX-Y.NYB) 수집 중...
Aluminum (ALI=F) 수집 중...

✅ 수집 완료: 714 행의 데이터가 저장되었습니다.
                     10Y_Bond         Gold  Copper  USD_Index  Aluminum
Datetime                                                               
2026-03-02 12:20:00     4.052  5381.200195  6.0020  98.545998    3155.0
2026-03-02 13:20:00     4.048  5381.200195  6.0020  98.545998    3155.0
2026-03-02 22:00:00     4.048  5381.200195  6.0020  98.545998    3155.0
2026-03-02 23:00:00     4.048  5377.899902  5.9795  98.637001    3155.0
2026-03-03 00:00:00     4.048  5331.899902  5.9210  98.725998    3104.0
2026-03-03 01:00:00     4.048  5323.000000  5.9495  98.696999    3108.0
2026-03-03 02:00:00     4.048  5340.600098  5.9345  98.879997    3114.0
2026-03-03 03:00:00     4.048  5308.799805  5.8015  99.040001    3093.5
2026-03-03 04:00:00     4.048  5286.799805  5.7905  99.160004    3110.0
2026-03-03 05:00:00     4.048  5259.700195  5.8110  99.28

In [ ]:
import re
from datetime import datetime, timedelta

def get_actual_time(time_text):
    """'2 hours ago' 같은 문자열을 실제 시간으로 변환합니다."""
    now = datetime.now()
    try:
        # 숫자만 추출
        number = int(re.search(r'\d+', time_text).group())
        if 'minute' in time_text:
            return (now - timedelta(minutes=number)).strftime('%Y-%m-%d %H:%M:%S')
        elif 'hour' in time_text:
            return (now - timedelta(hours=number)).strftime('%Y-%m-%d %H:%M:%S')
        elif 'day' in time_text:
            return (now - timedelta(days=number)).strftime('%Y-%m-%d %H:%M:%S')
    except:
        pass
    # 변환 실패 시 현재 시간 반환
    return now.strftime('%Y-%m-%d %H:%M:%S')

def fetch_yahoo_news_v2_updated(topic="economic-news"):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    url = f"https://finance.yahoo.com/topic/{topic}/"

    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        news_list = []
        articles = soup.find_all('h3')

        for title_tag in articles:
            link_tag = title_tag.find_parent('a') or title_tag.find('a')

            if link_tag and link_tag.get('href'):
                title = title_tag.get_text(strip=True)
                link = link_tag['href']
                if link.startswith('/'):
                    link = "https://finance.yahoo.com" + link

                # --- [수정된 부분: 시간 정보 추출] ---
                # 제목(h3)을 감싸고 있는 가장 가까운 부모 컨테이너(보통 li나 div)를 찾습니다.
                container = title_tag.find_parent(['li', 'div', 'section'])
                time_text = "Just now"

                if container:
                    # 컨테이너 안에서 'ago'라는 글자가 포함된 span이나 div를 찾습니다.
                    time_element = container.find(string=re.compile(r'ago|yesterday|hours|minutes'))
                    if time_element:
                        time_text = time_element.strip()

                # 상대 시간을 절대 시간으로 변환
                published_at = get_actual_time(time_text)
                # -------------------------------------

                if "video" not in link and len(title) > 10:
                    news_list.append({
                        "title": title,
                        "url": link,
                        "published_at": published_at, # collected_at 대신 사용
                        "source": "Yahoo Finance"
                    })

        news_df = pd.DataFrame(news_list).drop_duplicates(subset=['title'])

        if not news_df.empty:
            news_df.to_csv(f"{DATA_PATH}/raw_news.csv", index=False, encoding='utf-8-sig')
            print(f"성공: {len(news_df)}개의 뉴스 데이터를 수집했습니다.")
        else:
            print("경고: 수집된 뉴스가 없습니다.")

        return news_df

    except Exception as e:
        print(f"오류 발생: {e}")
        return pd.DataFrame()

# 실행
news_data = fetch_yahoo_news_v2_updated("economic-news")

성공: 58개의 뉴스 데이터를 수집했습니다.


In [ ]:
def fetch_yahoo_news_v2_4(topic="economic-news"):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36..."}
    url = f"https://finance.yahoo.com/topic/{topic}/"

    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        news_list = []
        articles = soup.find_all('h3')

        for title_tag in articles:
            link_tag = title_tag.find_parent('a') or title_tag.find('a')
            if not link_tag: continue

            title = title_tag.get_text(strip=True)
            link = link_tag['href']
            if link.startswith('/'): link = "https://finance.yahoo.com" + link

            # --- [수정된 시간 추출 로직] ---
            published_at = None

            # 1. 제목을 감싸는 가장 큰 부모를 찾습니다 (보통 한 섹션)
            container = title_tag.find_parent(['li', 'div', 'section'])

            if container:
                # 2. 컨테이너 내부의 모든 텍스트 중 'ago', 'hour', 'minute'가 포함된 요소를 찾습니다.
                # 직접적으로 태그를 지정하지 않고 텍스트 기반으로 검색합니다.
                time_elements = container.find_all(string=re.compile(r'ago|yesterday|minute|hour|day', re.I))

                for te in time_elements:
                    potential_time = te.strip()
                    # 너무 긴 문장은 제외 (예: "2 hours ago by Reuters")
                    if len(potential_time) < 30:
                        published_at = get_actual_time(potential_time)
                        break

            # 3. 여전히 못 찾았다면, 아주 드물게 나타나는 'time' 태그를 시도합니다.
            if not published_at:
                time_tag = container.find('time') if container else None
                if time_tag and time_tag.get('datetime'):
                    published_at = time_tag.get('datetime')[:19].replace('T', ' ')
                else:
                    # 마지막 수단: 정말 못 찾으면 현재 시간 (구분하기 위해 수집 시간임을 명시)
                    published_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            # -------------------------------

            if "video" not in link and len(title) > 10:
                news_list.append({
                    "title": title,
                    "url": link,
                    "published_at": published_at,
                    "source": "Yahoo Finance"
                })

        # 저장 및 출력 생략 (기존과 동일)
        return pd.DataFrame(news_list)

    except Exception as e:
        print(f"오류: {e}")
        return pd.DataFrame()

In [ ]:
# 시간 데이터가 얼마나 다양한지 확인
unique_times = news_data['published_at'].nunique()
print(f"수집된 뉴스 개수: {len(news_data)}")
print(f"서로 다른 시간의 개수: {unique_times}")

if unique_times == 1:
    print("❌ 여전히 모든 뉴스의 시간이 동일합니다. 추출 로직 보완이 필요합니다.")
else:
    print("✅ 뉴스의 시간이 분산되었습니다. 인과관계 분석이 가능합니다.")

수집된 뉴스 개수: 58
서로 다른 시간의 개수: 1
❌ 여전히 모든 뉴스의 시간이 동일합니다. 추출 로직 보완이 필요합니다.


In [ ]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 1.9 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=96498529762a28aac2cc04237b28d5fda8cf84bff5b02cce1ecc96417913be1e
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import feedparser
import pandas as pd
from datetime import datetime
import time

def fetch_yahoo_rss():
    # 야후 파이낸셜의 메인 뉴스 RSS 피드 주소
    rss_url = "https://finance.yahoo.com/news/rss"

    print(f"📡 RSS 피드 분석 중: {rss_url}")
    feed = feedparser.parse(rss_url)

    news_list = []

    for entry in feed.entries:
        # 1. 발행 시간 파싱 (RSS의 pubDate는 GMT 기준임)
        # feedparser는 시간 정보를 struct_time 객체로 자동 변환해줍니다.
        published_struct = entry.published_parsed
        published_at = time.strftime('%Y-%m-%d %H:%M:%S', published_struct)

        # 2. 데이터 구성
        news_list.append({
            "title": entry.title,
            "url": entry.link,
            "published_at": published_at,
            "source": "Yahoo Finance (RSS)"
        })

    # 데이터프레임 변환 및 저장
    news_df = pd.DataFrame(news_list)

    if not news_df.empty:
        # 인과관계 분석을 위해 시간순(최신순) 정렬
        news_df = news_df.sort_values(by='published_at', ascending=False)
        news_df.to_csv(f"{DATA_PATH}/raw_news.csv", index=False, encoding='utf-8-sig')
        print(f"✅ 성공: {len(news_df)}개의 뉴스 데이터를 RSS로 수집했습니다.")

        # 시간 분산도 확인을 위한 샘플 출력
        print("\n--- 수집된 뉴스 시간 샘플 ---")
        print(news_df['published_at'].head(5))
    else:
        print("❌ 뉴스 데이터를 가져오지 못했습니다.")

    return news_df

# 실행
news_data = fetch_yahoo_rss()

📡 RSS 피드 분석 중: https://finance.yahoo.com/news/rss
✅ 성공: 43개의 뉴스 데이터를 RSS로 수집했습니다.

--- 수집된 뉴스 시간 샘플 ---
30    2026-04-02 16:42:53
42    2026-04-02 01:49:47
41    2026-04-02 01:46:43
39    2026-04-02 01:44:32
38    2026-04-02 01:42:38
Name: published_at, dtype: object


In [ ]:
unique_times = news_data['published_at'].nunique()
print(f"전체 뉴스: {len(news_data)}개 / 고유한 시간대: {unique_times}개")

if unique_times > 1:
    print("✨ 드디어 뉴스의 시간이 분산되었습니다! 이제 지수 변동과 매칭할 수 있습니다.")
else:
    print("⚠️ 여전히 시간이 동일하다면 RSS 서버의 일시적 지연일 수 있습니다.")

전체 뉴스: 43개 / 고유한 시간대: 43개
✨ 드디어 뉴스의 시간이 분산되었습니다! 이제 지수 변동과 매칭할 수 있습니다.


In [ ]:
def fetch_robust_market_data():
    tickers = {
        "10Y_Bond": "^TNX",
        "Gold": "GC=F",
        "Copper": "HG=F",
        "USD_Index": "DX-Y.NYB",
        "Aluminum": "ALI=F"
    }

    all_data = []

    # 1. 개별 데이터 수집
    for name, ticker in tickers.items():
        print(f"{name} ({ticker}) 수집 중...")
        asset = yf.Ticker(ticker)
        df = asset.history(period="1mo", interval="1h")

        if not df.empty:
            df = df[['Close']].rename(columns={'Close': name})
            # 시간대 제거 및 정시(00분)로 시간 맞추기 (시각화 및 매칭 편의성)
            df.index = df.index.tz_localize(None).floor('H')
            all_data.append(df)

    # 2. 데이터 통합 (중복된 인덱스 제거 후 병합)
    market_df = pd.concat(all_data, axis=1)
    # 동일한 시간에 여러 데이터가 있을 경우 첫 번째 값만 사용
    market_df = market_df[~market_df.index.duplicated(keep='first')]

    # 3. [핵심] 연속적인 시간 그리드 생성
    # 데이터의 시작점부터 끝점까지 1시간 간격의 완벽한 리스트를 만듭니다.
    full_index = pd.date_range(
        start=market_df.index.min(),
        end=market_df.index.max(),
        freq='H'
    )

    # 4. 빈 시간 채우기
    # 생성한 완벽한 시간 리스트에 기존 데이터를 맞추고(reindex),
    # 빈 칸은 직전 가격(ffill)으로, 그래도 비어있다면 다음 가격(bfill)으로 채웁니다.
    market_df = market_df.reindex(full_index).ffill().bfill()

    # 5. 인덱스 이름 정리 및 저장
    market_df.index.name = "Datetime"
    market_df.to_csv(f"{DATA_PATH}/market_prices.csv")

    print(f"\n✅ 정규화 완료: {len(market_df)} 행의 연속 데이터가 저장되었습니다.")
    return market_df

# 실행
market_prices = fetch_robust_market_data()

10Y_Bond (^TNX) 수집 중...


/tmp/ipykernel_1767/3376617530.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_1767/3376617530.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')


Gold (GC=F) 수집 중...
Copper (HG=F) 수집 중...


/tmp/ipykernel_1767/3376617530.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')


USD_Index (DX-Y.NYB) 수집 중...


/tmp/ipykernel_1767/3376617530.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_1767/3376617530.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_1767/3376617530.py:31: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(


Aluminum (ALI=F) 수집 중...

✅ 정규화 완료: 756 행의 연속 데이터가 저장되었습니다.


In [ ]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.9 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=a91429133fde1ea0927aa411e2450f25d9a38fc37f3642d476202a73a2411364
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import feedparser
import pandas as pd
import yfinance as yf
from datetime import datetime
import time
import os

# [보완] 다각화된 RSS 소스 리스트
RSS_SOURCES = {
    "Yahoo_Main": "https://finance.yahoo.com/news/rss",
    "Yahoo_CentralBank": "https://finance.yahoo.com/news/category-central-banks/rss",
    "Yahoo_Economy": "https://finance.yahoo.com/news/category-economy/rss"
}

def fetch_accumulated_news():
    all_news = []

    for name, url in RSS_SOURCES.items():
        print(f"📡 {name} 피드 분석 중...")
        feed = feedparser.parse(url)
        for entry in feed.entries:
            published_struct = entry.published_parsed
            published_at = time.strftime('%Y-%m-%d %H:%M:%S', published_struct)

            all_news.append({
                "title": entry.title,
                "url": entry.link,
                "summary": entry.get('summary', ''), # [추가] 요약 내용 포함 (RAG 성능 향상)
                "published_at": published_at,
                "source": name
            })

    new_df = pd.DataFrame(all_news)

    # [핵심] 누적 저장 로직
    file_path = f"{DATA_PATH}/raw_news.csv"
    if os.path.exists(file_path):
        old_df = pd.read_csv(file_path)
        combined_df = pd.concat([old_df, new_df])
        # URL 기준으로 중복 제거 (이미 수집한 뉴스는 제외)
        final_df = combined_df.drop_duplicates(subset=['url'], keep='first')
    else:
        final_df = new_df

    final_df.sort_values(by='published_at', ascending=False, inplace=True)
    final_df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"✅ 총 {len(final_df)}개의 뉴스 데이터가 저장소에 확보되었습니다. (신규: {len(new_df)})")
    return final_df

def fetch_robust_market_data(period="1mo"): # 기간을 파라미터로 조절 가능하게 변경
    tickers = {
        "10Y_Bond": "^TNX",
        "Gold": "GC=F",
        "Copper": "HG=F",
        "USD_Index": "DX-Y.NYB",
        "Aluminum": "ALI=F"
    }

    all_data = []
    for name, ticker in tickers.items():
        print(f"📈 {name} ({ticker}) 수집 중...")
        try:
            asset = yf.Ticker(ticker)
            df = asset.history(period=period, interval="1h")
            if not df.empty:
                df = df[['Close']].rename(columns={'Close': name})
                df.index = df.index.tz_localize(None).floor('H')
                all_data.append(df)
        except Exception as e:
            print(f"⚠️ {name} 수집 중 오류: {e}")

    market_df = pd.concat(all_data, axis=1)
    market_df = market_df[~market_df.index.duplicated(keep='first')]

    # 타임 그리드 정규화
    full_index = pd.date_range(start=market_df.index.min(), end=market_df.index.max(), freq='H')
    market_df = market_df.reindex(full_index).ffill().bfill()

    market_df.index.name = "Datetime"
    market_df.to_csv(f"{DATA_PATH}/market_prices.csv")
    print(f"✅ 마켓 데이터 정규화 완료: {len(market_df)} 행")
    return market_df

# 실행부
news_data = fetch_accumulated_news()
market_data = fetch_robust_market_data(period="1mo")

📡 Yahoo_Main 피드 분석 중...
📡 Yahoo_CentralBank 피드 분석 중...
📡 Yahoo_Economy 피드 분석 중...


NameError: name 'DATA_PATH' is not defined

In [ ]:
import feedparser
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import time
import os

# ==========================================
# 1. 전역 설정 (Global Configuration)
# ==========================================
# 구글 드라이브 내 영구 저장 경로 설정
DATA_PATH = "/content/drive/MyDrive/Mecro/data"

if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)
    print(f"✅ 구글 드라이브 경로 확인: {DATA_PATH}")

# 다각화된 RSS 소스 리스트
RSS_SOURCES = {
    "Yahoo_Main": "https://finance.yahoo.com/news/rss",
    "Yahoo_CentralBank": "https://finance.yahoo.com/news/category-central-banks/rss",
    "Yahoo_Economy": "https://finance.yahoo.com/news/category-economy/rss"
}

# ==========================================
# 2. 뉴스 수집 함수 (RSS 기반 - Title & Time 중심)
# ==========================================
def fetch_accumulated_news():
    all_news = []

    for name, url in RSS_SOURCES.items():
        print(f"📡 {name} 피드 분석 중...")
        feed = feedparser.parse(url)
        for entry in feed.entries:
            # 시간 파싱
            try:
                published_struct = entry.published_parsed
                published_at = time.strftime('%Y-%m-%d %H:%M:%S', published_struct)
            except:
                published_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            # [수정] summary를 제외하고 핵심 메타데이터만 수집
            all_news.append({
                "title": entry.title,
                "url": entry.link,
                "published_at": published_at,
                "source": name
            })

    new_df = pd.DataFrame(all_news)

    file_path = f"{DATA_PATH}/raw_news.csv"

    # 누적 저장 로직: 기존 데이터와 병합 및 중복 제거
    if os.path.exists(file_path):
        old_df = pd.read_csv(file_path)
        combined_df = pd.concat([old_df, new_df])
        # URL 기준으로 중복된 기사 제거
        final_df = combined_df.drop_duplicates(subset=['url'], keep='first')
    else:
        final_df = new_df

    # 시간순 정렬 후 저장
    final_df.sort_values(by='published_at', ascending=False, inplace=True)
    final_df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"✅ 뉴스 지식 베이스 업데이트 완료: 총 {len(final_df)}개 (신규: {len(new_df)})")
    return final_df

# ==========================================
# 3. 마켓 지수 수집 함수 (시계열 정규화)
# ==========================================
def fetch_robust_market_data(period="1mo"):
    tickers = {
        "10Y_Bond": "^TNX",
        "Gold": "GC=F",
        "Copper": "HG=F",
        "USD_Index": "DX-Y.NYB",
        "Aluminum": "ALI=F"
    }

    all_data = []
    for name, ticker in tickers.items():
        print(f"📈 {name} ({ticker}) 데이터 로드 중...")
        try:
            asset = yf.Ticker(ticker)
            df = asset.history(period=period, interval="1h")
            if not df.empty:
                df = df[['Close']].rename(columns={'Close': name})
                # 시간대 제거 및 정시 단위 정규화
                df.index = df.index.tz_localize(None).floor('H')
                all_data.append(df)
        except Exception as e:
            print(f"⚠️ {name} 수집 실패: {e}")

    if not all_data:
        return pd.DataFrame()

    # 데이터 병합 및 중복 시간 제거
    market_df = pd.concat(all_data, axis=1)
    market_df = market_df[~market_df.index.duplicated(keep='first')]

    # 타임 그리드 생성 (비어있는 시간대 없이 촘촘하게 연결)
    full_index = pd.date_range(start=market_df.index.min(), end=market_df.index.max(), freq='H')
    market_df = market_df.reindex(full_index).ffill().bfill()

    market_df.index.name = "Datetime"
    market_df.to_csv(f"{DATA_PATH}/market_prices.csv")
    print(f"✅ 마켓 시계열 정규화 완료: {len(market_df)} 행 확보")
    return market_df

# ==========================================
# 4. 실행 (Factory 가동)
# ==========================================
news_data = fetch_accumulated_news()
market_data = fetch_robust_market_data(period="1mo")

📡 Yahoo_Main 피드 분석 중...
📡 Yahoo_CentralBank 피드 분석 중...
📡 Yahoo_Economy 피드 분석 중...
✅ 뉴스 지식 베이스 업데이트 완료: 총 50개 (신규: 50)
📈 10Y_Bond (^TNX) 데이터 로드 중...
📈 Gold (GC=F) 데이터 로드 중...


/tmp/ipykernel_8995/2410669008.py:90: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_8995/2410669008.py:90: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_8995/2410669008.py:90: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_8995/2410669008.py:90: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')


📈 Copper (HG=F) 데이터 로드 중...
📈 USD_Index (DX-Y.NYB) 데이터 로드 중...
📈 Aluminum (ALI=F) 데이터 로드 중...
✅ 마켓 시계열 정규화 완료: 732 행 확보


/tmp/ipykernel_8995/2410669008.py:90: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.index = df.index.tz_localize(None).floor('H')
/tmp/ipykernel_8995/2410669008.py:103: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(start=market_df.index.min(), end=market_df.index.max(), freq='H')
